In [3]:
!pip install -q transformers accelerate ftfy diffusers

In [8]:
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'  # we can never escape

In [7]:
import torch
import torch.nn as nn
from diffusers import AutoencoderKLWan, WanPipeline

from taehv import TAEHV

class DotDict(dict):
    __getattr__ = dict.__getitem__
    __setattr__ = dict.__setitem__

class TAEW2_2DiffusersWrapper(nn.Module):
    def __init__(self):
        super().__init__()
        self.dtype = torch.float16
        self.device = "cuda"
        self.taehv = TAEHV("taew2_2.pth").to(self.dtype)
        self.config = DotDict(
            scaling_factor=1.0,
            latents_mean=torch.zeros(self.taehv.latent_channels),
            z_dim=self.taehv.latent_channels,
            latents_std=torch.ones(self.taehv.latent_channels)
        )

    def decode(self, latents, return_dict=None):
        n, c, t, h, w = latents.shape
        # low-memory, set parallel=True for faster + higher memory
        return (self.taehv.decode_video(latents.transpose(1, 2), parallel=False).transpose(1, 2).mul_(2).sub_(1),)

model_id = "Wan-AI/Wan2.2-TI2V-5B-Diffusers"
vae = AutoencoderKLWan.from_pretrained(model_id, subfolder="vae", torch_dtype=torch.float32)
pipe = WanPipeline.from_pretrained(model_id, vae=vae, torch_dtype=torch.bfloat16)
pipe.vae = TAEW2_2DiffusersWrapper().to("cuda")
pipe.to("cuda")

The config attributes {'clip_output': False} were passed to AutoencoderKLWan, but are not expected and will be ignored. Please verify your config.json configuration file.


Loading pipeline components...:   0%|          | 0/5 [00:00<?, ?it/s]

Loading checkpoint shards:   0%|          | 0/5 [00:00<?, ?it/s]

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

WanPipeline {
  "_class_name": "WanPipeline",
  "_diffusers_version": "0.35.1",
  "_name_or_path": "Wan-AI/Wan2.2-TI2V-5B-Diffusers",
  "boundary_ratio": null,
  "expand_timesteps": true,
  "scheduler": [
    "diffusers",
    "UniPCMultistepScheduler"
  ],
  "text_encoder": [
    "transformers",
    "UMT5EncoderModel"
  ],
  "tokenizer": [
    "transformers",
    "T5TokenizerFast"
  ],
  "transformer": [
    "diffusers",
    "WanTransformer3DModel"
  ],
  "transformer_2": [
    null,
    null
  ],
  "vae": [
    "__main__",
    "TAEW2_2DiffusersWrapper"
  ]
}

In [9]:
output = pipe(
    prompt="a spinning slice of delicious new-york style cheesecake with mint and berries is placed onto a plate and then drizzled with berry sauce",
    negative_prompt="Bright tones, overexposed, static, blurred details, subtitles, style, works, paintings, images, static, overall gray, worst quality, low quality, JPEG compression residue, ugly, incomplete, extra fingers, poorly drawn hands, poorly drawn faces, deformed, disfigured, misshapen limbs, fused fingers, still picture, messy background, three legs, many people in the background, walking backwards",
    height=704,
    width=1280,
    num_frames=81,
    guidance_scale=5.0,
    generator=torch.Generator("cpu").manual_seed(0x7AE),
    output_type="pil"
).frames[0]

  0%|          | 0/50 [00:00<?, ?it/s]

/home/ubuntu/.local/lib/python3.10/site-packages/diffusers/pipelines/wan/pipeline_wan.py:637: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  torch.tensor(self.vae.config.latents_mean)
/home/ubuntu/.local/lib/python3.10/site-packages/diffusers/pipelines/wan/pipeline_wan.py:641: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  latents_std = 1.0 / torch.tensor(self.vae.config.latents_std).view(1, self.vae.config.z_dim, 1, 1, 1).to(


  0%|          | 0/21 [00:00<?, ?it/s]

In [19]:
from IPython.display import HTML
import numpy as np
import cv2
import subprocess as sp

class VideoImageWriter:
    def __init__(self, video_file_path, width_height, fps=30):
        self.writer = cv2.VideoWriter(video_file_path, cv2.VideoWriter_fourcc(*'mp4v'), fps, width_height)
        assert self.writer.isOpened(), f"Could not create writer for {video_file_path}"
    def write(self, image):
        self.writer.write(cv2.cvtColor(np.array(image), cv2.COLOR_RGB2BGR)) # RGB CHW -> BGR HWC
    def __del__(self):
        if hasattr(self, 'writer'): self.writer.release()

def frames_to_html(frames):
    """Display TCHW [0, 1] frame tensor as inline video on colab."""
    out_path = "taew2_2_demo.mp4"
    out_compressed_path = "taew2_2_demo_compressed.mp4"
    out = VideoImageWriter(out_path, frames[0].size, fps=15)
    for frame in frames:
        out.write(frame)
    del out
    ffmpeg_command = [
        "ffmpeg",
        "-y",  # Overwrite output files without asking
        "-i", out_path,  # Input file
        "-hide_banner",  # Suppress startup banner
        "-loglevel", "error",  # Set log level to error
        "-c:v", "libx264",  # Set video codec to libx264
        "-crf", "10",
        "-threads", "0",  # Use all available threads
        "-pix_fmt", "yuv420p",  # Set pixel format
        out_compressed_path  # Output file
    ]
    sp.run(ffmpeg_command)
    return HTML(f"<video loop width={frame.width} autoplay muted><source src='{out_compressed_path}'></source></video>")

display(frames_to_html(output))